# 17. SQL Transactions: ACID & Concurrency Isolation: Beginner Guide

### 📝 SQL Execution Order for ACID Transactions:
```text
┌─ Execution Order ────────────────────────────────────────────────────────────┐
│ 1. BEGIN (Open Snapshot) ➔ 2. DML STREAM (Buffer) ➔ 3. SAVEPOINT Checkpoints │
│ ➔ 4. VALIDATE ISOLATION (No Anomalies) ➔ 5. COMMIT (WAL Flush) or ROLLBACK   │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **17. SQL Transactions: ACID & Concurrency Isolation**. Relational databases guarantee reliable data operations through ACID properties: Atomicity, Consistency, Isolation, and Durability. This notebook covers explicit transaction boundaries (`BEGIN`, `COMMIT`, `ROLLBACK`), granular checkpoints (`SAVEPOINT`), SQL standard isolation levels, and concurrency anomalies (Dirty Reads, Non-Repeatable Reads, Phantom Reads).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 ACID Properties: Atomicity, Consistency, Isolation, Durability
- [x] 🔹 Transaction Boundaries: `BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK`
- [x] 🔹 Granular Savepoints: `SAVEPOINT point_name` & `ROLLBACK TO SAVEPOINT`
- [x] 🔹 Isolation Levels: `READ COMMITTED` to `SERIALIZABLE`
- [x] 🔍 Scenario: Atomically Executing a Multi-Account Balance Payout & Rollback









In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Transaction Boundaries: `BEGIN`, `COMMIT`, `ROLLBACK`
- **What it does:** Enforces atomicity: either all operations in the batch execute successfully (`COMMIT`) or all intermediate mutations are cleanly undone (`ROLLBACK`).
- **Syntax:** `BEGIN TRANSACTION; UPDATE ...; UPDATE ...; COMMIT;`
- **Dataset Application & Code Demonstration:** Simulates an atomic fund transfer between two customer accounts.


In [2]:
%%sql
CREATE TABLE IF NOT EXISTS account_balances (
    account_id TEXT PRIMARY KEY,
    balance REAL CHECK (balance >= 0)
);
INSERT OR REPLACE INTO account_balances VALUES ('ACC_A', 1000.00), ('ACC_B', 500.00);

BEGIN TRANSACTION;
UPDATE account_balances SET balance = balance - 200.00 WHERE account_id = 'ACC_A';
UPDATE account_balances SET balance = balance + 200.00 WHERE account_id = 'ACC_B';
COMMIT;

SELECT * FROM account_balances;


'Query Executed Successfully.'

### 🔹 Granular Savepoints: `SAVEPOINT` & `ROLLBACK TO`
- **What it does:** Sets an internal checkpoint within a transaction, allowing partial rollback without aborting the entire transaction.
- **Syntax:** `SAVEPOINT sp1; ... ROLLBACK TO sp1; ... RELEASE sp1;`
- **Dataset Application & Code Demonstration:** Uses savepoints to rollback an invalid balance withdrawal.


In [3]:
%%sql
BEGIN TRANSACTION;
UPDATE account_balances SET balance = balance + 50.00 WHERE account_id = 'ACC_A';
SAVEPOINT intermediate_checkpoint;
UPDATE account_balances SET balance = balance + 9999.00 WHERE account_id = 'ACC_B';
ROLLBACK TO intermediate_checkpoint;
COMMIT;

SELECT * FROM account_balances;


'Query Executed Successfully.'

## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Concurrency Isolation Levels & Anomalies Matrix
- **Objective:** Analyze the 4 SQL standard isolation levels and the specific concurrency anomalies each level prevents.
- **Approach:** Construct an enterprise transaction isolation decision matrix.


In [4]:
%%sql
SELECT 
    'READ UNCOMMITTED' AS isolation_level, 'Yes' AS dirty_read, 'Yes' AS non_repeatable_read, 'Yes' AS phantom_read
UNION ALL
SELECT 'READ COMMITTED', 'No', 'Yes', 'Yes'
UNION ALL
SELECT 'REPEATABLE READ', 'No', 'No', 'Yes'
UNION ALL
SELECT 'SERIALIZABLE', 'No', 'No', 'No';


,isolation_level,dirty_read,non_repeatable_read,phantom_read
0,READ UNCOMMITTED,Yes,Yes,Yes
1,READ COMMITTED,No,Yes,Yes
2,REPEATABLE READ,No,No,Yes
3,SERIALIZABLE,No,No,No
